In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
torch.cuda.is_available()

True

# Generate data

In [3]:
import random

NUM_ITEMS = 8  # item ids will be 1..NUM_ITEMS -- 0 stays reserved for padding
BASE_PRICE = {item_id: random.randint(500, 3000) for item_id in range(1, NUM_ITEMS + 1)}

def generate_customer(min_events=3, max_events=8, min_age=20, max_age=60):
    n_events = random.randint(min_events, max_events)
    age = random.randint(min_age, max_age)

    items, ages, prices = [], [], []
    for _ in range(n_events):
        item_id = random.randint(1, NUM_ITEMS)
        price = BASE_PRICE[item_id] * random.lognormvariate(0, 0.15)

        items.append(item_id)
        ages.append(age)
        prices.append(round(price, 2))

        age += random.randint(0, 3)  # customer ages a bit between purchases

    return items, ages, prices

def generate_dataset(n_customers=500, min_events=3, max_events=8):
    return [generate_customer(min_events, max_events) for _ in range(n_customers)]

In [4]:
samples = generate_dataset(n_customers=10_000)
len(samples)

10000

In [5]:
samples[0], samples[-1]

(([7, 4, 2, 6, 4, 5, 3],
  [21, 22, 22, 25, 27, 27, 30],
  [3101.06, 806.64, 1202.62, 2578.05, 877.2, 1609.17, 3452.76]),
 ([5, 2, 4, 7, 8, 1],
  [58, 61, 61, 61, 64, 66],
  [1440.07, 782.53, 810.64, 3384.18, 1283.66, 1625.81]))

# Start!

In [6]:
import numpy as np
import pandas as pd

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

In [8]:
from sequences_to_multiple_outcomes.dataset import BaseDataset
from sequences_to_multiple_outcomes.collate_functions import next_seq_collate_fn

In [9]:
dataset = BaseDataset(data=samples)

n_total = len(dataset)
n_train = int(n_total * .7)
n_val = int(n_total * .15)
n_test = n_total - n_train - n_val

train_set, val_set, test_set = random_split(
    # this approach can be used with Dataset subclass?
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(55)
)

len(train_set), len(val_set), len(test_set)

(7000, 1500, 1500)

In [10]:
BATCH_SIZE = 1024
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=next_seq_collate_fn)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=next_seq_collate_fn)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=next_seq_collate_fn)

In [11]:
MAX_SEQ_LEN = len(max(samples, key=lambda x: len(x[0]))[0])
MAX_SEQ_LEN

8

In [12]:
class Normalizer:
    def __init__(self):
        self.mean: float | None = None
        self.std: float | None = None
    
    def fit(self, X: torch.Tensor):
        self.mean = X.mean().item()
        self.std = X.std().item()
        return self

    def transform(self, X: torch.Tensor) -> torch.Tensor:
        return (X - self.mean) / self.std

    def inverse_transform(self, X: torch.Tensor) -> torch.Tensor:
        return (X * self.std) + self.mean

def collect_train_values(base_dataset, train_subset, field_idx):
    values = []
    for idx in train_subset.indices:
        values.extend(base_dataset.data[idx][field_idx])
    return torch.tensor(values, dtype=torch.float)

In [13]:
age_normalizer = Normalizer().fit(collect_train_values(dataset, train_set, field_idx=1))
price_normalizer = Normalizer().fit(collect_train_values(dataset, train_set, field_idx=2))

In [39]:
from sequences_to_multiple_outcomes.encoder_bock import EncoderCLS
from sequences_to_multiple_outcomes.embedding_layers import FusionEmbedding, FusionStrategy
from sequences_to_multiple_outcomes.positional_embedding_layers import RoPE, RoPEStrategy
from sequences_to_multiple_outcomes.loss_functions import UncertaintyWeightedLoss, TaskType
from sequences_to_multiple_outcomes.downstream_tasks import MultiTaskModel
from dataclasses import dataclass, field

@dataclass
class ExperimentConfig:
    d_model: int
    max_seq_len: int
    n_heads: int
    expansion: int
    num_layers: int
    fusion_strategy: FusionStrategy
    rope_strategy: RoPEStrategy

    max_len: int = field(init=False)
    head_dim: int = field(init=False) # use this with RoPE because MultiAttention converts (d_model) to (n_heads, head_dim)

    def __post_init__(self):
        """self.max_len is for RoPE"""
        self.max_len = self.max_seq_len + 1  # +1 for CLS token
        self.head_dim = self.d_model // self.n_heads

d_model = 256
n_heads = 4
config = ExperimentConfig(
    d_model=d_model,
    max_seq_len=MAX_SEQ_LEN,
    n_heads=n_heads,
    expansion=4,
    num_layers=4,
    fusion_strategy=FusionStrategy.CONCAT,
    rope_strategy=RoPEStrategy.INTERLEAVED,
)

fusion = FusionEmbedding(
    categorical_vocab_sizes=[NUM_ITEMS+1], 
    num_continuous=2, 
    d_model=config.d_model, 
    strategy=config.fusion_strategy
)

rope = RoPE(config.head_dim, config.max_len, strategy=config.rope_strategy)

encoder_block = EncoderCLS(
    fusion = fusion,
    position_emb = rope,
    d_model = config.d_model,
    n_heads = config.n_heads,
    num_layers = config.num_layers,
    expansion = config.expansion,
)

model = MultiTaskModel(
    backbone = encoder_block,
    d_model = config.d_model,
    vocab_size = NUM_ITEMS+1
)

In [40]:
# encoder_block(categorical_features=[items], continuous_features=[ages, prices], pad_mask=mask).shape

In [41]:
# pred_item, pred_age, pred_price = model(categorical_features=[items], continuous_features=[ages, prices], pad_mask=mask)

In [42]:
def normalize_batch(ages, prices, next_age, next_price):
    return (
        age_normalizer.transform(ages),
        price_normalizer.transform(prices),
        age_normalizer.transform(next_age),
        price_normalizer.transform(next_price)
    )

In [43]:
loss_weigher = UncertaintyWeightedLoss(task_types=[TaskType.CLASSIFICATION, TaskType.REGRESSION, TaskType.REGRESSION])
optimizer = torch.optim.Adam(list(model.parameters())+list(loss_weigher.parameters()), lr=1e-3)

EPOCHS = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
loss_weigher = loss_weigher.to(device)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for items, ages, prices, next_item, next_age, next_price, mask in train_loader:
        items, ages, prices = items.to(device), ages.to(device), prices.to(device)
        next_item, next_age, next_price = next_item.to(device), next_age.to(device), next_price.to(device)
        mask = mask.to(device)
        ages_norm, prices_norm, next_age_norm, next_price_norm = normalize_batch(
            ages, prices, next_age, next_price
        )
        pred_item, pred_age, pred_price = model(
            categorical_features=[items], 
            continuous_features=[ages_norm, prices_norm], 
            pad_mask=mask
        )
        loss_item = F.cross_entropy(pred_item, next_item)
        loss_age = F.mse_loss(pred_age.squeeze(-1), next_age_norm)
        loss_price = F.mse_loss(pred_price.squeeze(-1), next_price_norm)
        loss = loss_weigher([loss_item, loss_age, loss_price])
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")
        

Epoch 10, Loss: 20.9910
Epoch 20, Loss: 20.2714
Epoch 30, Loss: 19.6719
Epoch 40, Loss: 19.0774
Epoch 50, Loss: 18.4387
Epoch 60, Loss: 17.9515
Epoch 70, Loss: 17.5136
Epoch 80, Loss: 17.1006
Epoch 90, Loss: 16.7171
Epoch 100, Loss: 16.3575


In [56]:
pred_item.argmax(dim=-1).int()

tensor([4, 5, 2, 5, 4, 5, 5, 2, 5, 5, 2, 5, 2, 5, 2, 5, 5, 4, 5, 5, 5, 2, 2, 2,
        2, 4, 5, 2, 5, 2, 5, 5, 4, 5, 2, 2, 5, 5, 5, 5, 2, 4, 2, 4, 5, 2, 2, 4,
        2, 2, 5, 4, 5, 4, 2, 4, 4, 2, 2, 2, 2, 2, 5, 5, 2, 5, 2, 4, 5, 5, 4, 5,
        2, 2, 4, 5, 2, 2, 4, 2, 5, 2, 4, 2, 5, 2, 4, 2, 2, 4, 2, 2, 5, 5, 4, 4,
        2, 2, 2, 2, 4, 2, 2, 5, 5, 2, 2, 2, 5, 2, 5, 2, 2, 5, 4, 2, 2, 2, 2, 5,
        2, 2, 4, 2, 4, 5, 4, 5, 5, 2, 5, 5, 5, 5, 5, 2, 4, 5, 4, 4, 5, 5, 5, 4,
        2, 5, 2, 2, 5, 2, 2, 4, 2, 5, 5, 5, 2, 5, 4, 4, 2, 5, 4, 2, 2, 2, 4, 2,
        2, 4, 2, 2, 2, 5, 5, 2, 4, 2, 5, 2, 4, 5, 2, 2, 2, 2, 5, 5, 2, 5, 4, 4,
        5, 5, 4, 2, 2, 2, 2, 2, 5, 4, 2, 2, 2, 2, 5, 5, 5, 5, 4, 5, 2, 4, 2, 5,
        5, 4, 5, 2, 5, 5, 2, 5, 2, 2, 4, 2, 5, 2, 4, 4, 4, 2, 5, 2, 5, 2, 2, 2,
        2, 5, 5, 2, 5, 5, 2, 2, 2, 2, 4, 2, 5, 4, 2, 5, 2, 2, 2, 2, 5, 2, 4, 5,
        5, 5, 5, 4, 2, 5, 5, 5, 5, 5, 5, 2, 2, 2, 5, 5, 4, 2, 4, 5, 2, 4, 5, 5,
        5, 5, 5, 4, 5, 2, 2, 2, 4, 5, 2,

In [66]:
freq = pd.crosstab(next_item.int().cpu().numpy(), pred_item.argmax(dim=-1).int().cpu().numpy())
freq.loc[:, list(set(freq.index) - set(freq.columns))] = 0
freq.loc[:, [i+1 for i in range(freq.shape[0])]]

col_0,1,2,3,4,5,6,7,8
row_0,,,,,,,,
1,0,54,0,24,32,0,0,0
2,0,31,0,28,39,0,0,0
3,0,47,0,26,37,0,0,0
4,0,40,0,27,46,0,0,0
5,0,33,0,23,43,0,0,0
6,0,46,0,21,40,0,0,0
7,0,45,0,19,45,0,0,0
8,0,45,0,19,46,0,0,0


In [57]:
next_item.int()

tensor([8, 4, 1, 1, 3, 1, 2, 2, 3, 7, 1, 4, 2, 3, 4, 4, 7, 5, 8, 1, 1, 4, 6, 6,
        3, 2, 7, 7, 6, 7, 4, 2, 7, 5, 3, 1, 3, 8, 4, 4, 2, 7, 4, 7, 3, 8, 6, 1,
        1, 2, 1, 1, 4, 2, 4, 4, 1, 7, 2, 4, 4, 8, 7, 5, 1, 3, 8, 7, 6, 7, 7, 6,
        6, 7, 2, 6, 8, 6, 4, 7, 7, 2, 7, 2, 8, 1, 1, 6, 1, 3, 6, 8, 1, 5, 3, 5,
        3, 4, 5, 6, 5, 8, 4, 6, 8, 5, 2, 1, 6, 8, 5, 1, 1, 6, 5, 5, 1, 3, 4, 6,
        2, 8, 5, 3, 3, 5, 2, 3, 7, 1, 7, 4, 8, 2, 2, 8, 7, 4, 4, 1, 2, 4, 3, 6,
        1, 6, 6, 6, 1, 5, 7, 2, 3, 4, 1, 7, 3, 8, 2, 6, 5, 4, 5, 8, 8, 5, 2, 4,
        6, 2, 5, 8, 4, 6, 2, 3, 8, 4, 5, 2, 5, 7, 1, 5, 6, 7, 6, 4, 5, 8, 8, 2,
        4, 6, 3, 1, 4, 7, 8, 8, 1, 4, 2, 2, 2, 8, 4, 8, 4, 3, 5, 4, 3, 2, 4, 1,
        2, 2, 6, 8, 5, 3, 8, 2, 1, 1, 5, 8, 4, 6, 7, 7, 5, 1, 7, 8, 6, 7, 3, 6,
        1, 1, 8, 7, 2, 5, 5, 3, 7, 6, 7, 6, 3, 3, 4, 3, 6, 3, 3, 5, 6, 1, 2, 5,
        6, 4, 8, 7, 1, 8, 2, 2, 8, 5, 1, 7, 2, 6, 5, 5, 4, 2, 5, 3, 3, 4, 6, 2,
        2, 2, 3, 8, 5, 4, 8, 8, 4, 2, 3,

In [53]:
age_normalizer.inverse_transform(pred_age.squeeze(-1)).int()

tensor([52, 37, 59, 26, 66, 32, 24, 35, 31, 55, 59, 29, 48, 42, 63, 53, 46, 44,
        22, 24, 29, 63, 60, 57, 58, 28, 32, 59, 50, 56, 27, 46, 30, 31, 65, 62,
        57, 62, 33, 65, 40, 33, 58, 45, 58, 42, 57, 30, 64, 47, 47, 55, 48, 28,
        60, 55, 40, 58, 59, 63, 44, 60, 27, 31, 62, 28, 50, 37, 37, 51, 61, 46,
        59, 61, 40, 31, 71, 52, 30, 39, 46, 51, 54, 54, 24, 66, 48, 48, 55, 53,
        28, 57, 27, 61, 47, 31, 50, 52, 45, 38, 47, 33, 30, 44, 33, 54, 63, 38,
        47, 46, 49, 38, 67, 64, 44, 42, 43, 64, 48, 31, 62, 49, 35, 69, 50, 32,
        36, 40, 52, 42, 34, 42, 48, 22, 39, 57, 43, 27, 59, 25, 66, 35, 42, 41,
        67, 47, 64, 51, 36, 35, 36, 33, 54, 26, 25, 40, 42, 55, 42, 59, 34, 28,
        43, 55, 33, 56, 46, 47, 63, 36, 54, 64, 63, 28, 65, 37, 36, 65, 60, 61,
        38, 47, 50, 26, 63, 49, 29, 47, 63, 49, 43, 37, 29, 55, 64, 65, 40, 55,
        58, 39, 41, 38, 65, 65, 36, 51, 54, 34, 25, 38, 54, 30, 37, 57, 72, 28,
        47, 44, 30, 67, 32, 73, 61, 34, 

In [55]:
next_age.int()

tensor([51, 36, 59, 28, 65, 34, 24, 36, 32, 55, 61, 31, 47, 44, 64, 55, 48, 45,
        21, 24, 28, 65, 60, 59, 57, 27, 34, 61, 49, 55, 27, 46, 32, 33, 65, 61,
        56, 62, 35, 67, 42, 35, 57, 47, 59, 41, 56, 31, 65, 49, 46, 57, 47, 30,
        61, 57, 42, 57, 61, 65, 46, 59, 28, 32, 61, 27, 49, 38, 38, 53, 63, 47,
        60, 62, 42, 33, 72, 52, 29, 41, 45, 51, 53, 54, 25, 65, 50, 48, 57, 53,
        30, 57, 27, 62, 48, 32, 52, 52, 46, 39, 49, 32, 29, 43, 33, 54, 63, 39,
        49, 46, 49, 40, 68, 63, 45, 42, 45, 66, 48, 31, 61, 50, 37, 69, 49, 34,
        35, 41, 54, 44, 35, 41, 48, 21, 41, 58, 43, 29, 60, 27, 67, 34, 44, 42,
        66, 47, 65, 52, 35, 34, 38, 34, 55, 26, 26, 40, 43, 56, 43, 58, 36, 29,
        43, 56, 34, 55, 46, 49, 62, 36, 54, 64, 65, 30, 66, 39, 38, 66, 59, 61,
        37, 47, 52, 25, 65, 49, 29, 47, 65, 51, 43, 39, 29, 57, 65, 67, 41, 55,
        59, 38, 43, 39, 65, 67, 36, 53, 53, 36, 24, 40, 53, 29, 39, 58, 71, 27,
        47, 45, 30, 69, 32, 73, 63, 36, 